# Experiment 2 (Enriched Index): Frozen SapBERT Baseline — With Translation

**Goal:** Evaluate frozen SapBERT (`cambridgeltl/SapBERT-from-PubMedBERT-fulltext`) on cross-lingual biomedical entity linking against an ICD-11 concept index.  
**Condition:** Traditional Chinese surface forms are used with conversion to Simplified Chinese 
**Corpora:** `english_ncbi`, `simp_chinese`, `trad_chinese`  
**Model:** Completely frozen throughout — `model.eval()` + `torch.no_grad()` everywhere, no optimizer, no gradient updates.

## GPU Check

In [20]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. Please enable a GPU runtime in Colab: "
        "Runtime -> Change runtime type -> Hardware accelerator -> GPU."
    )

print(f"GPU available: {torch.cuda.get_device_name(0)}")
print(f"CUDA version:  {torch.version.cuda}")

GPU available: Tesla T4
CUDA version:  12.8


## Step 1 — Setup

Install dependencies and configure paths.

In [21]:
!pip install -q transformers faiss-cpu

In [22]:
# Mount Google Drive so we can read the corpus file.
# After running this cell, authorise access in the popup.
from google.colab import drive
drive.mount('/content/drive')

# Verify the mount succeeded
import os
print(os.listdir('/content/drive/MyDrive'))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['Getting started.pdf', '5.\xa0Acceptance\xa0of\xa0Project\xa0-\xa0_80\xa0(_3.5M).pdf', '5.\xa0Acceptance\xa0of\xa0Project\xa0-\xa0_80\xa0(_3.5M).gdoc', 'UCSD SOP.docx', 'Q.Q from drive', 'Berkeley', 'GENKI 1.zip', 'opt-20220603T170706Z-001.zip', 'R notebook-20220603T171726Z-001.zip', 'qpcr templates-20220603T171726Z-001.zip', 'Protocols-20220603T171721Z-001.zip', 'rsem_script(1)', 'rsem_script', 'guides', 'Weichen_Rsem_scripts', 'Windsor Village at Waltham - 5203-quote.pdf', 'PowerDVD18.0.1815.62（极致蓝光版）.zip', 'DVDFab12.01.7.X64.zip', 'Weichen Zhao resume.gdoc', 'COSI103A SWE Project', 'Workout plan.gdoc', 'ALC 080 LP_K1 Package_Weichen Zhao_2024_02-24-25.pdf', 'Copy of Agile Project Infographics by Slidesgo.gslides', 'Copy of Elegant Lines Pitch Deck | by Slidesgo.gslides', '2025_BIONER_project', 'Zhao_Weichen_resume_2025_fAIshion.pdf', 'Colab Notebooks', 'H

In [23]:
import sys
import transformers
import faiss
import numpy as np
import pandas as pd

print(f"Python:       {sys.version.split()[0]}")
print(f"torch:        {torch.__version__}")
print(f"transformers: {transformers.__version__}")
print(f"faiss:        {faiss.__version__}")
print(f"numpy:        {np.__version__}")
print(f"pandas:       {pd.__version__}")

Python:       3.12.13
torch:        2.10.0+cu128
transformers: 5.0.0
faiss:        1.13.2
numpy:        2.0.2
pandas:       2.2.2


In [24]:
import os

# -----------------------------------------------------------------------
# Set REPO_ROOT to the folder in your Google Drive that contains the corpus
# -----------------------------------------------------------------------
REPO_ROOT = "/content/drive/MyDrive/2026_Medical_Entity_Linking_project/"  # <-- adjust if needed

DATA_PATH = os.path.join(
    REPO_ROOT,
    "data/with_translation/v1/combined_disease_corpus_train_with_cuis_icd11_cleaned.jsonl"
)

OUTPUT_DIR    = os.path.join(REPO_ROOT, "experiments/frozen_sapbert_with_translation/v1")
os.makedirs(OUTPUT_DIR, exist_ok=True)

RESULTS_JSONL = os.path.join(OUTPUT_DIR, "baseline_results_exp2_enriched_index.jsonl")
SUMMARY_CSV   = os.path.join(OUTPUT_DIR, "baseline_summary_exp2_enriched_index.csv")

SAPBERT_MODEL = "cambridgeltl/SapBERT-from-PubMedBERT-fulltext"
BATCH_SIZE    = 32
DEVICE        = torch.device("cuda")

# Sanity-check that the data file exists before going further
assert os.path.exists(DATA_PATH), (
    f"Data file not found: {DATA_PATH}\n"
    "Check that REPO_ROOT is set correctly and the file exists in Google Drive."
)

print(f"Data path:  {DATA_PATH}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"Device:     {DEVICE}")

Data path:  /content/drive/MyDrive/2026_Medical_Entity_Linking_project/data/with_translation/v1/combined_disease_corpus_train_with_cuis_icd11_cleaned.jsonl
Output dir: /content/drive/MyDrive/2026_Medical_Entity_Linking_project/experiments/frozen_sapbert_with_translation/v1
Device:     cuda


## Step 2 — Load and Filter Data

In [25]:
import json
from collections import defaultdict

evaluable = []   # entities with a non-null ontology_id
skipped   = []   # entities with ontology_id == null

with open(DATA_PATH, "r", encoding="utf-8") as fh:
    for line in fh:
        line = line.strip()
        if not line:
            continue
        record = json.loads(line)
        corpus = record["source_corpus"]
        for ent in record.get("entities", []):
            row = {
                "entity_id":      ent["entity_id"],
                "surface_form":   ent["surface_form"],
                "source_corpus":  corpus,
                "ontology_id":    ent.get("ontology_id"),
                "ontology_label": ent.get("ontology_label"),
            }
            if ent.get("ontology_id") is None:
                skipped.append(row)
            else:
                evaluable.append(row)

def corpus_counts(records):
    counts = defaultdict(int)
    for r in records:
        counts[r["source_corpus"]] += 1
    return dict(counts)

print(f"Evaluable entities (non-null ontology_id): {len(evaluable):,}")
print(f"  by corpus: {corpus_counts(evaluable)}")
print()
print(f"Skipped entities (null ontology_id): {len(skipped):,}")
print(f"  by corpus: {corpus_counts(skipped)}")

Evaluable entities (non-null ontology_id): 21,389
  by corpus: {'english_ncbi': 2158, 'simp_chinese': 13425, 'trad_chinese': 5806}

Skipped entities (null ontology_id): 9,011
  by corpus: {'english_ncbi': 797, 'simp_chinese': 5946, 'trad_chinese': 2268}


### Index Construction: Single-Label vs Enriched

| | Single-label (original) | Enriched (this notebook) |
|---|---|---|
| Vectors per URI | 1 (canonical label only) | 1 + all training surface forms |
| FAISS index size | = number of unique URIs | ≥ number of unique URIs |
| Retrieval | Top-10 vectors = top-10 URIs | Top-10 unique URIs after dedup |
| Mention coverage | Canonical labels only | Canonical + seen surface forms |

**Motivation:** The original SapBERT paper (Liu et al., 2021) trains on UMLS, whose concept index is rich with synonyms — each CUI has multiple preferred terms, aliases, and cross-lingual variants. Adding all training-set surface forms as additional index vectors gives the retriever more anchor points per concept, which should particularly help for Chinese mentions whose script differs from the canonical English ICD-11 label.

> Note: enrichment uses only the training split, so there is no leakage — test surface forms are not added to the index.

## Step 3 — Build Enriched Concept Index

For each unique URI collect:
- The canonical `ontology_label` from the data
- All distinct `surface_form` values that appear in the training set with that URI

Each text representation is encoded separately with frozen SapBERT (mean pooling → L2 normalise).  
All vectors for a URI are added to the FAISS `IndexFlatIP` index, so the index contains **more vectors than unique URIs**.  
A mapping `faiss_int → ontology_id` enables URI retrieval for any vector.

In [26]:
from transformers import AutoTokenizer, AutoModel

print(f"Loading SapBERT from '{SAPBERT_MODEL}' ...")
tokenizer = AutoTokenizer.from_pretrained(SAPBERT_MODEL)
model     = AutoModel.from_pretrained(SAPBERT_MODEL).to(DEVICE)

model.eval()
for param in model.parameters():
    param.requires_grad = False

print("Model loaded and completely frozen.")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

Loading SapBERT from 'cambridgeltl/SapBERT-from-PubMedBERT-fulltext' ...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: cambridgeltl/SapBERT-from-PubMedBERT-fulltext
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded and completely frozen.
Total parameters: 109,482,240


In [27]:
def encode_texts(texts, batch_size=BATCH_SIZE):
    """Return L2-normalised float32 embeddings for a list of strings."""
    all_embeddings = []
    model.eval()
    with torch.no_grad():
        for start in range(0, len(texts), batch_size):
            batch = texts[start : start + batch_size]
            encoded = tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=64,
                return_tensors="pt",
            ).to(DEVICE)
            output = model(**encoded)
            # Mean pooling over token positions
            attention_mask   = encoded["attention_mask"].unsqueeze(-1).float()  # (B, T, 1)
            token_embeddings = output.last_hidden_state                          # (B, T, H)
            summed    = (token_embeddings * attention_mask).sum(dim=1)           # (B, H)
            counts    = attention_mask.sum(dim=1)                                # (B, 1)
            mean_pooled = summed / counts                                        # (B, H)
            all_embeddings.append(mean_pooled.cpu().float().numpy())
    embeddings = np.concatenate(all_embeddings, axis=0)  # (N, H)
    # L2 normalise
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    norms = np.maximum(norms, 1e-12)
    return (embeddings / norms).astype(np.float32)

In [28]:
# Build per-URI text representation sets
# Canonical label always first; training surface forms appended.
uri_to_canonical = {}   # ontology_id -> canonical ICD-11 label
uri_to_surfaces  = {}   # ontology_id -> set of training surface forms

for ent in evaluable:
    oid   = ent["ontology_id"]
    label = ent["ontology_label"] or ""
    sf    = ent["surface_form"]
    if oid not in uri_to_canonical:
        uri_to_canonical[oid] = label
        uri_to_surfaces[oid]  = set()
    uri_to_surfaces[oid].add(sf)

# Build flat parallel lists for encoding
# index_to_id[i] = ontology_id for FAISS vector i
all_texts   = []
index_to_id = []

for oid in uri_to_canonical:
    canonical = uri_to_canonical[oid]
    surfaces  = uri_to_surfaces[oid]
    texts_for_uri = [canonical] + [s for s in sorted(surfaces) if s != canonical]
    for t in texts_for_uri:
        all_texts.append(t)
        index_to_id.append(oid)

# ── Summary stats ────────────────────────────────────────────────────────
uri_vec_counts = {}
for oid in index_to_id:
    uri_vec_counts[oid] = uri_vec_counts.get(oid, 0) + 1

counts = list(uri_vec_counts.values())
import numpy as _np
bins = {"1": 0, "2-5": 0, "6-10": 0, "10+": 0}
for c in counts:
    if   c == 1:  bins["1"]    += 1
    elif c <= 5:  bins["2-5"]  += 1
    elif c <= 10: bins["6-10"] += 1
    else:         bins["10+"]  += 1

print(f"Unique URIs:              {len(uri_vec_counts):,}")
print(f"Total vectors in index:   {len(all_texts):,}")
print(f"Avg vectors per URI:      {_np.mean(counts):.2f}")
print(f"URIs with 1 surface form:    {bins['1']:,}")
print(f"URIs with 2-5 surface forms: {bins['2-5']:,}")
print(f"URIs with 6-10 surface forms:{bins['6-10']:,}")
print(f"URIs with 10+ surface forms: {bins['10+']:,}")

# ── Encode and build FAISS index ─────────────────────────────────────────
print("\nEncoding all concept representations with frozen SapBERT ...")
concept_embeddings = encode_texts(all_texts)
print(f"Concept embeddings shape: {concept_embeddings.shape}")

dim   = concept_embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(concept_embeddings)
print(f"FAISS IndexFlatIP built with {index.ntotal:,} vectors (dim={dim}).")


Unique URIs:              3,168
Total vectors in index:   7,011
Avg vectors per URI:      2.21
URIs with 1 surface form:    515
URIs with 2-5 surface forms: 2,612
URIs with 6-10 surface forms:37
URIs with 10+ surface forms: 4

Encoding all concept representations with frozen SapBERT ...
Concept embeddings shape: (7011, 768)
FAISS IndexFlatIP built with 7,011 vectors (dim=768).


## Step 4 — Encode Mentions

Encode each evaluable entity's `surface_form` with the same frozen SapBERT encoder.  
**Experiment 2 condition:** Traditional Chinese surface forms are used with OpenCC translation.

In [29]:
print(f"Encoding {len(evaluable):,} mention surface forms with frozen SapBERT ...")

surface_forms      = [ent["surface_form"] for ent in evaluable]
mention_embeddings = encode_texts(surface_forms)

print(f"Mention embeddings shape: {mention_embeddings.shape}")

Encoding 21,389 mention surface forms with frozen SapBERT ...
Mention embeddings shape: (21389, 768)


## Step 5 — Retrieve and Evaluate

Retrieve top-10 candidates per mention from the FAISS index.  
Compute Acc@1, Acc@5, Acc@10 for the full dataset and per source corpus.

In [30]:
K        = 10      # desired number of unique URIs per mention
K_FETCH  = K * 10  # over-fetch to ensure K unique URIs after dedup

print(f"Searching FAISS index (fetch {K_FETCH} vectors, dedup to top-{K} unique URIs) "
      f"for {len(evaluable):,} mentions ...")
_scores, _indices = index.search(mention_embeddings, K_FETCH)  # (N, K_FETCH)
print("Search complete.")


Searching FAISS index (fetch 100 vectors, dedup to top-10 unique URIs) for 21,389 mentions ...
Search complete.


In [31]:
per_entity_results = []

for i, ent in enumerate(evaluable):
    gold_uri = ent["ontology_id"]

    # Deduplicate by URI while preserving rank order
    seen      = set()
    retrieved = []
    for idx in _indices[i]:
        uri = index_to_id[idx]
        if uri not in seen:
            seen.add(uri)
            retrieved.append(uri)
        if len(retrieved) == K:
            break

    top1_correct  = retrieved[0] == gold_uri
    top5_correct  = gold_uri in retrieved[:5]
    top10_correct = gold_uri in retrieved[:10]

    per_entity_results.append({
        "entity_id":          ent["entity_id"],
        "surface_form":       ent["surface_form"],
        "source_corpus":      ent["source_corpus"],
        "gold_uri":           gold_uri,
        "top1_predicted_uri": retrieved[0],
        "top1_correct":       top1_correct,
        "top5_correct":       top5_correct,
        "top10_correct":      top10_correct,
        "top5_candidates":    retrieved[:5],
    })

print(f"Evaluation complete for {len(per_entity_results):,} entities.")


Evaluation complete for 21,389 entities.


In [32]:
def compute_acc(records):
    n = len(records)
    if n == 0:
        return {"N": 0, "Acc@1": float("nan"), "Acc@5": float("nan"), "Acc@10": float("nan")}
    return {
        "N":      n,
        "Acc@1":  round(sum(r["top1_correct"]  for r in records) / n, 4),
        "Acc@5":  round(sum(r["top5_correct"]  for r in records) / n, 4),
        "Acc@10": round(sum(r["top10_correct"] for r in records) / n, 4),
    }

rows = {}
for corpus in ["english_ncbi", "simp_chinese", "trad_chinese"]:
    subset = [r for r in per_entity_results if r["source_corpus"] == corpus]
    rows[corpus] = compute_acc(subset)
rows["ALL"] = compute_acc(per_entity_results)

summary_df = pd.DataFrame(rows).T[["N", "Acc@1", "Acc@5", "Acc@10"]]
summary_df.index.name = "corpus"

print("\n=== Experiment 2 (Enriched Index): Frozen SapBERT — With Translation (Enriched Index) ===")
print(summary_df.to_string())


=== Experiment 2 (Enriched Index): Frozen SapBERT — With Translation (Enriched Index) ===
                    N   Acc@1   Acc@5  Acc@10
corpus                                       
english_ncbi   2158.0  0.5667  0.6622  0.7312
simp_chinese  13425.0  0.6282  0.7791  0.8007
trad_chinese   5806.0  0.5429  0.7113  0.7589
ALL           21389.0  0.5988  0.7489  0.7823


In [33]:
summary_df

,N,Acc@1,Acc@5,Acc@10
corpus,,,,
english_ncbi,2158.0,0.5667,0.6622,0.7312
simp_chinese,13425.0,0.6282,0.7791,0.8007
trad_chinese,5806.0,0.5429,0.7113,0.7589
ALL,21389.0,0.5988,0.7489,0.7823


## Step 6 — Save Results

In [34]:
# Per-entity JSONL
with open(RESULTS_JSONL, "w", encoding="utf-8") as fh:
    for row in per_entity_results:
        fh.write(json.dumps(row, ensure_ascii=False) + "\n")
print(f"Per-entity results saved to: {RESULTS_JSONL}")

# Summary CSV
summary_df.to_csv(SUMMARY_CSV)
print(f"Summary table saved to:      {SUMMARY_CSV}")

Per-entity results saved to: /content/drive/MyDrive/2026_Medical_Entity_Linking_project/experiments/frozen_sapbert_with_translation/v1/baseline_results_exp2_enriched_index.jsonl
Summary table saved to:      /content/drive/MyDrive/2026_Medical_Entity_Linking_project/experiments/frozen_sapbert_with_translation/v1/baseline_summary_exp2_enriched_index.csv


---
## Next: Experiment 3: Fine-tuning